In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


# Load Data

df = pd.read_excel("business_regression_data.xlsx")


# Create Dummy Variables

df_model = pd.get_dummies(
    df,
    columns=["region", "store_type"],
    drop_first=True,
    dtype=int
)


# Select Independent Variables

predictors = [
    'marketing_spend',
    'footfall',
    'inventory_availability_pct',
    'customer_rating'
]

dummy_cols = [col for col in df_model.columns
              if col.startswith('region_') or col.startswith('store_type_')]

predictors += dummy_cols


# Build Dataset

X = df_model[predictors]
y = df_model['monthly_sales']

# Replace inf with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Combine X and y and remove rows with missing values
combined = pd.concat([X, y], axis=1).dropna()

# Separate again
X = combined.drop(columns=['monthly_sales'])
y = combined['monthly_sales']

# Add intercept
X = sm.add_constant(X)


# Run Regression

model = sm.OLS(y, X).fit()

print(model.summary())


# Coefficient Table

coef_df = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": model.params.values,
    "P_value": model.pvalues.values
})

print("\nCoefficients:")
print(coef_df)


# Predictions and Residuals

predicted_sales = model.predict(X)

residual_df = pd.DataFrame({
    "Actual_Sales": y,
    "Predicted_Sales": predicted_sales,
    "Residual": y - predicted_sales
})

print("\nResidual Preview:")
print(residual_df.head())


# Save into Workbook

with pd.ExcelWriter(
    "regression_workbook.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    coef_df.to_excel(
        writer,
        sheet_name="Multiple_Regression_Output",
        index=False
    )

    residual_df.to_excel(
        writer,
        sheet_name="Predictions_Residuals",
        index=False
    )

print("\nWorkbook updated successfully!")

print("\nR² =", round(model.rsquared,4))
print("Adjusted R² =", round(model.rsquared_adj,4))

                            OLS Regression Results                            
Dep. Variable:          monthly_sales   R-squared:                       0.828
Model:                            OLS   Adj. R-squared:                  0.822
Method:                 Least Squares   F-statistic:                     144.7
Date:                Mon, 22 Jun 2026   Prob (F-statistic):          1.18e-108
Time:                        12:17:39   Log-Likelihood:                -3766.7
No. Observations:                 312   AIC:                             7555.
Df Residuals:                     301   BIC:                             7597.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const               